# MCS-201 Hardware Acceleration
**Artificial Intelligence and System Engineering · Prince of Songkla University, Phuket Campus**

---

# Lab 4 — Optimising Your Own Kernel
### Shared memory tiling, coalescing, bank conflicts และการปรับ launch configuration

| | |
|---|---|
| **สัปดาห์** | 5 |
| **วันปฏิบัติการ** | จันทร์ 14 ก.ย. 2569 |
| **กำหนดส่ง** | จันทร์ 14 ก.ย. 2569 **ภายใน 13.00 น.** (วันเดียวกับคาบปฏิบัติการ) |
| **น้ำหนัก** | อยู่ในหมวด Lab ซึ่งคิดเป็น 15% ของคะแนนรายวิชา (ทุกแลปถ่วงน้ำหนักเท่ากัน) |
| **ช่องทางส่ง** | LMS — lms.psu.ac.th |
| **ไฟล์ที่ต้องส่ง** | `Lab4_<รหัสนักศึกษา>.ipynb` พร้อมไฟล์ `.cu` (บีบเป็น .zip) |
| **ผู้สอน** | ผศ.ดร.ฐิตินันท์ เกลี้ยงสุวรรณ · thitinan.kl@phuket.psu.ac.th · ห้อง 6714 |

> **ต้องใช้ผลจาก Lab 3** แลปนี้วัด speedup เทียบกับ `matmul` แบบ naive ที่ท่านเขียนเองในสัปดาห์ที่แล้ว
> หากยังไม่มีตัวเลข baseline ให้รันโค้ดของ Lab 3 ซ้ำก่อนเริ่มส่วนที่ 1


## 1. วัตถุประสงค์ (Objectives)

1. เขียน tiled matrix multiplication ด้วย shared memory และวาง `__syncthreads()` ได้ถูกตำแหน่ง — *CLO2*
2. อธิบายและ **วัด** ผลของ memory coalescing ที่มีต่อแบนด์วิดท์ที่ทำได้จริง — *CLO2, CLO3*
3. สร้าง shared memory bank conflict ขึ้นมาเอง แล้วแก้ด้วยการ padding พร้อมรายงานอัตราส่วน — *CLO3*
4. อ่านการใช้ register/shared memory จาก `-Xptxas -v` และเลือก block size จากการวัด ไม่ใช่จากการเดา — *CLO2, CLO3*

## 2. การคอมไพล์บน Google Colab

รูปแบบเดียวกับ Lab 3 — ใช้ `%%writefile` เขียนไฟล์ `.cu` แล้วเรียก `nvcc` ผ่าน shell

> **สำคัญ** ทุกการจับเวลาในแลปนี้ใช้ **CUDA events** และต้องมี warm-up ก่อนเสมอ
> ตัวเลขที่ได้จาก `time.time()` ของ Python ใช้ไม่ได้ (เหตุผลอยู่ใน Lab 3 ส่วนที่ 2)


In [13]:
!nvcc --version
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
name, compute_cap, memory.total [MiB]
Tesla T4, 7.5, 15360 MiB


In [14]:
ARCH = "sm_75"      # TODO: แก้ให้ตรงกับ GPU ของท่าน
                    # sm_75 = T4 · sm_80 = A100 · sm_89 = L4 · sm_90 = H100

# TODO: กรอกค่า baseline จาก Lab 3 ของท่านเอง (matmul แบบ naive)
BASE_N2048_MS = 37.815    # เวลา (ms) ของ matmul naive ที่ N = 2048
print("using", ARCH, "· baseline =", BASE_N2048_MS, "ms")

using sm_75 · baseline = 37.815 ms


---
## 3. ขั้นตอนการปฏิบัติ

### ส่วนที่ 1 — Tiled matmul (T = 16)

เติม `TODO` ทั้ง 5 จุด โปรแกรมนี้รัน **ทั้ง naive และ tiled** ในไฟล์เดียว
ตรวจความถูกต้องกับผลที่คำนวณด้วยมือก่อน แล้วจึงดูเวลา

> A ทุกช่อง = 1.0 และ B ทุกช่อง = 2.0 ดังนั้นทุกช่องของ C ต้องเท่ากับ `2N` พอดี


In [15]:
%%writefile matmul_tiled.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CUDA_CHECK(call)                                        \
  do {                                                          \
    cudaError_t err = (call);                                   \
    if (err != cudaSuccess) {                                   \
      fprintf(stderr, "CUDA %s:%d: %s\n", __FILE__, __LINE__,   \
              cudaGetErrorString(err));                         \
      exit(EXIT_FAILURE);                                       \
    }                                                           \
  } while (0)

#define T 16

/* ---- baseline จาก Lab 3 (ให้มาแล้ว ห้ามแก้) ---------------------- */
__global__ void matmulNaive(int N, const float *A, const float *B, float *C) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) sum += A[row*N + k] * B[k*N + col];
        C[row*N + col] = sum;
    }
}

/* ---- ส่วนที่ต้องเขียนเอง ---------------------------------------- */
__global__ void matmulTiled(int N, const float *A, const float *B, float *C) {
    __shared__ float As[T][T];
    __shared__ float Bs[T][T];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * T + ty;
    int col = blockIdx.x * T + tx;
    float sum = 0.0f;

    for (int t = 0; t < (N + T - 1) / T; t++) {
        int aCol = t * T + tx;
        int bRow = t * T + ty;

        /* TODO (1): โหลด A หนึ่งช่องลง As[ty][tx]
                     ถ้า row หรือ aCol เกินขอบเขต ให้ใส่ 0.0f */
        As[ty][tx] = (row < N && aCol < N) ? A[row*N + aCol] : 0.0f;

        /* TODO (2): โหลด B หนึ่งช่องลง Bs[ty][tx] ในทำนองเดียวกัน */
        Bs[ty][tx] = (bRow < N && col < N) ? B[bRow*N + col] : 0.0f;

        /* TODO (3): barrier ตัวที่หนึ่ง — กันไม่ให้ใครอ่านไทล์ก่อนเขียนเสร็จ */
        __syncthreads();

        for (int k = 0; k < T; k++) {
            /* TODO (4): สะสมผลคูณจาก shared memory ลงใน sum */
            sum += As[ty][k] * Bs[k][tx];
        }

        /* TODO (5): barrier ตัวที่สอง — กันไม่ให้ใครเขียนทับไทล์ที่ยังมีคนอ่าน */
        __syncthreads();
    }
    if (row < N && col < N) C[row*N + col] = sum;
}

/* ---- host code (ให้มาแล้ว) -------------------------------------- */
static float timeKernel(void (*launch)(int, const float*, const float*, float*),
                        int N, const float *dA, const float *dB, float *dC) {
    cudaEvent_t s, e; float ms = 0.0f;
    CUDA_CHECK(cudaEventCreate(&s)); CUDA_CHECK(cudaEventCreate(&e));
    launch(N, dA, dB, dC);                       /* warm-up */
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaEventRecord(s));
    for (int r = 0; r < 5; r++) launch(N, dA, dB, dC);
    CUDA_CHECK(cudaEventRecord(e));
    CUDA_CHECK(cudaEventSynchronize(e));
    CUDA_CHECK(cudaEventElapsedTime(&ms, s, e));
    CUDA_CHECK(cudaEventDestroy(s)); CUDA_CHECK(cudaEventDestroy(e));
    return ms / 5.0f;
}

static void launchNaive(int N, const float *A, const float *B, float *C) {
    dim3 blk(T, T), grd((N + T - 1)/T, (N + T - 1)/T);
    matmulNaive<<<grd, blk>>>(N, A, B, C);
}
static void launchTiled(int N, const float *A, const float *B, float *C) {
    dim3 blk(T, T), grd((N + T - 1)/T, (N + T - 1)/T);
    matmulTiled<<<grd, blk>>>(N, A, B, C);
}

int main(int argc, char **argv) {
    int N = (argc > 1) ? atoi(argv[1]) : 1024;
    size_t bytes = (size_t)N * N * sizeof(float);

    float *hA = (float*)malloc(bytes), *hB = (float*)malloc(bytes),
          *hC = (float*)malloc(bytes);
    for (int i = 0; i < N*N; i++) { hA[i] = 1.0f; hB[i] = 2.0f; }

    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, bytes)); CUDA_CHECK(cudaMalloc(&dB, bytes));
    CUDA_CHECK(cudaMalloc(&dC, bytes));
    CUDA_CHECK(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));

    float tn = timeKernel(launchNaive, N, dA, dB, dC);
    float tt = timeKernel(launchTiled, N, dA, dB, dC);

    CUDA_CHECK(cudaMemcpy(hC, dC, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaGetLastError());
    double expect = 2.0 * N, maxerr = 0.0;
    for (int i = 0; i < N*N; i++) maxerr = fmax(maxerr, fabs(hC[i] - expect));

    double gflop = 2.0 * N * N * N / 1e9;
    printf("N = %d   ตรวจความถูกต้อง: ค่าที่ควรได้ %.1f  ค่าคลาดเคลื่อนสูงสุด %.3f  %s\n",
           N, expect, maxerr, (maxerr < 1e-2) ? "[ผ่าน]" : "[ไม่ผ่าน]");
    printf("  naive : %8.3f ms   %6.3f TFLOPS\n", tn, gflop / tn);
    printf("  tiled : %8.3f ms   %6.3f TFLOPS   speedup = %.2fx\n",
           tt, gflop / tt, tn / tt);

    free(hA); free(hB); free(hC);
    CUDA_CHECK(cudaFree(dA)); CUDA_CHECK(cudaFree(dB)); CUDA_CHECK(cudaFree(dC));
    return 0;
}

Overwriting matmul_tiled.cu


In [16]:
!nvcc -O3 -arch={ARCH} matmul_tiled.cu -o matmul_tiled
!./matmul_tiled 512
!./matmul_tiled 1024
!./matmul_tiled 2048

N = 512   ตรวจความถูกต้อง: ค่าที่ควรได้ 1024.0  ค่าคลาดเคลื่อนสูงสุด 0.000  [ผ่าน]
  naive :    0.798 ms    0.336 TFLOPS
  tiled :    0.514 ms    0.522 TFLOPS   speedup = 1.55x
N = 1024   ตรวจความถูกต้อง: ค่าที่ควรได้ 2048.0  ค่าคลาดเคลื่อนสูงสุด 0.000  [ผ่าน]
  naive :    4.509 ms    0.476 TFLOPS
  tiled :    2.810 ms    0.764 TFLOPS   speedup = 1.60x
N = 2048   ตรวจความถูกต้อง: ค่าที่ควรได้ 4096.0  ค่าคลาดเคลื่อนสูงสุด 0.000  [ผ่าน]
  naive :   37.243 ms    0.461 TFLOPS
  tiled :   22.255 ms    0.772 TFLOPS   speedup = 1.67x


---
### ส่วนที่ 2 — ไทล์ 32 และการทดลองเรื่อง coalescing

สองการเปลี่ยนแปลง วัดทีละอย่าง

1. **ไทล์ 32** — ใช้ `T = 32` (1024 เธรดต่อ block) ทำให้แต่ละไทล์ถูกใช้ซ้ำ 32 ครั้งแทน 16
2. **สลับแกน** — เปลี่ยน `row` ให้ผูกกับ `threadIdx.x` และ `col` ผูกกับ `threadIdx.y`
   ตรรกะยังถูกต้องทุกประการ แต่รูปแบบการเข้าถึงหน่วยความจำเปลี่ยนไปทั้งหมด

> เดาก่อนวัด — เขียนตัวเลขที่ท่านคาดไว้ลงในเซลล์คำตอบ **ก่อน** กดรัน


In [17]:
%%writefile matmul_axis.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CUDA_CHECK(call)                                        \
  do {                                                          \
    cudaError_t err = (call);                                   \
    if (err != cudaSuccess) {                                   \
      fprintf(stderr, "CUDA %s:%d: %s\n", __FILE__, __LINE__,   \
              cudaGetErrorString(err));                         \
      exit(EXIT_FAILURE);                                       \
    }                                                           \
  } while (0)

#define T 32

/* (ก) ไทล์ 32 แบบปกติ: tx -> คอลัมน์ (แกนที่เปลี่ยนเร็วที่สุด) */
__global__ void tiled32(int N, const float *A, const float *B, float *C) {
    __shared__ float As[T][T], Bs[T][T];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y*T + ty, col = blockIdx.x*T + tx;
    float sum = 0.0f;
    for (int t = 0; t < N/T; t++) {
        As[ty][tx] = A[row*N + (t*T + tx)];
        Bs[ty][tx] = B[(t*T + ty)*N + col];
        __syncthreads();
        for (int k = 0; k < T; k++) sum += As[ty][k] * Bs[k][tx];
        __syncthreads();
    }
    C[row*N + col] = sum;
}

/* (ข) สลับแกน: tx -> แถว  ผลลัพธ์ถูกต้องเหมือนเดิมทุกประการ */
__global__ void tiled32Swapped(int N, const float *A, const float *B, float *C) {
    __shared__ float As[T][T], Bs[T][T];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y*T + tx;            /* <-- สลับ */
    int col = blockIdx.x*T + ty;            /* <-- สลับ */
    float sum = 0.0f;
    for (int t = 0; t < N/T; t++) {
        As[tx][ty] = A[row*N + (t*T + ty)];
        Bs[tx][ty] = B[(t*T + tx)*N + col];
        __syncthreads();
        for (int k = 0; k < T; k++) sum += As[tx][k] * Bs[k][ty];
        __syncthreads();
    }
    C[row*N + col] = sum;
}

static float timeIt(int which, int N, const float *dA, const float *dB, float *dC) {
    dim3 blk(T, T), grd(N/T, N/T);
    cudaEvent_t s, e; float ms = 0.0f;
    CUDA_CHECK(cudaEventCreate(&s)); CUDA_CHECK(cudaEventCreate(&e));
    for (int r = 0; r < 6; r++) {
        if (r == 1) CUDA_CHECK(cudaEventRecord(s));   /* r = 0 คือ warm-up */
        if (which == 0) tiled32<<<grd, blk>>>(N, dA, dB, dC);
        else            tiled32Swapped<<<grd, blk>>>(N, dA, dB, dC);
    }
    CUDA_CHECK(cudaEventRecord(e));
    CUDA_CHECK(cudaEventSynchronize(e));
    CUDA_CHECK(cudaEventElapsedTime(&ms, s, e));
    CUDA_CHECK(cudaGetLastError());
    return ms / 5.0f;
}

int main(int argc, char **argv) {
    int N = (argc > 1) ? atoi(argv[1]) : 2048;
    size_t bytes = (size_t)N * N * sizeof(float);
    float *hA = (float*)malloc(bytes), *hB = (float*)malloc(bytes),
          *hC = (float*)malloc(bytes);
    for (int i = 0; i < N*N; i++) { hA[i] = 1.0f; hB[i] = 2.0f; }
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, bytes)); CUDA_CHECK(cudaMalloc(&dB, bytes));
    CUDA_CHECK(cudaMalloc(&dC, bytes));
    CUDA_CHECK(cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice));

    float t0 = timeIt(0, N, dA, dB, dC);
    float t1 = timeIt(1, N, dA, dB, dC);
    CUDA_CHECK(cudaMemcpy(hC, dC, bytes, cudaMemcpyDeviceToHost));
    double expect = 2.0 * N, maxerr = 0.0;
    for (int i = 0; i < N*N; i++) maxerr = fmax(maxerr, fabs(hC[i] - expect));
    double gflop = 2.0 * N * N * N / 1e9;

    printf("N = %d   ค่าคลาดเคลื่อนสูงสุด %.3f  %s\n",
           N, maxerr, (maxerr < 1e-2) ? "[ผ่าน]" : "[ไม่ผ่าน]");
    printf("  tiled32          : %8.3f ms   %6.3f TFLOPS\n", t0, gflop/t0);
    printf("  tiled32 สลับแกน  : %8.3f ms   %6.3f TFLOPS   ช้าลง %.2f เท่า\n",
           t1, gflop/t1, t1/t0);
    free(hA); free(hB); free(hC);
    CUDA_CHECK(cudaFree(dA)); CUDA_CHECK(cudaFree(dB)); CUDA_CHECK(cudaFree(dC));
    return 0;
}

Overwriting matmul_axis.cu


In [18]:
!nvcc -O3 -arch={ARCH} matmul_axis.cu -o matmul_axis
!./matmul_axis 2048

N = 2048   ค่าคลาดเคลื่อนสูงสุด 0.000  [ผ่าน]
  tiled32          :   16.978 ms    1.012 TFLOPS
  tiled32 สลับแกน  :  111.043 ms    0.155 TFLOPS   ช้าลง 6.54 เท่า


**บันทึกผลและตอบคำถาม**

(ก) ก่อนรัน ท่านคาดว่าเวอร์ชันสลับแกนจะช้าลงกี่เท่า และวัดได้จริงกี่เท่า

_ตอบ:_ คาดว่าช้าลงประมาณ 4 เท่า เพราะอ่านแบบ stride N ส่วนผลจริงคือ **6.54 เท่า**

(ข) เหตุใดการสลับแกนจึงทำให้ช้าลง ให้อธิบายโดยระบุว่า **เธรดที่ติดกันใน warp เดียวกัน**
เข้าถึงที่อยู่ใดบ้างในบรรทัด `As[tx][ty] = A[row*N + ty ...]` เทียบกับเวอร์ชันปกติ

_ตอบ:_ แบบปกติ warp อ่าน `A[row*N+t*T+0..31]` และ B ในคอลัมน์ติดกัน จึง coalesced แต่แบบสลับแกน เธรดติดกันอ่าน A และ B ห่างกันครั้งละ N float ทำให้ใช้ memory transaction มากขึ้นและช้าลง


---
### ส่วนที่ 3 — Bank conflict ที่วัดได้จริง (shared-memory transpose)

Matmul แบบ tiled ในส่วนที่ 1–2 **ไม่มี** bank conflict (`As[ty][k]` เป็น broadcast ส่วน `Bs[k][tx]`
กระจายครบ 32 bank) การจะเห็น conflict ต้องมีการอ่าน **ตามแนวตั้ง** ของไทล์ ซึ่งกรณีคลาสสิกคือ
**matrix transpose**

โปรแกรมด้านล่างมี transpose สามเวอร์ชัน ทั้งสามให้ผลลัพธ์เหมือนกันทุกประการ

| เวอร์ชัน | เขียน/อ่าน global | ไทล์ shared |
|---|---|---|
| `transposeNaive` | อ่าน coalesced · เขียน stride N | ไม่ใช้ |
| `transposeTile` | coalesced ทั้งสองทาง | `[32][32]` |
| `transposeTilePad` | coalesced ทั้งสองทาง | `[32][33]` |


In [19]:
%%writefile transpose.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CUDA_CHECK(call)                                        \
  do {                                                          \
    cudaError_t err = (call);                                   \
    if (err != cudaSuccess) {                                   \
      fprintf(stderr, "CUDA %s:%d: %s\n", __FILE__, __LINE__,   \
              cudaGetErrorString(err));                         \
      exit(EXIT_FAILURE);                                       \
    }                                                           \
  } while (0)

#define TILE 32
#define ROWS 8              /* แต่ละเธรดจัดการ 4 แถว: 32x8 = 256 เธรดต่อ block */

/* (ก) ไม่ใช้ shared memory เลย: การเขียนมี stride = N */
__global__ void transposeNaive(int N, const float *in, float *out) {
    int x = blockIdx.x*TILE + threadIdx.x;
    int y = blockIdx.y*TILE + threadIdx.y;
    for (int j = 0; j < TILE; j += ROWS)
        out[x*N + (y+j)] = in[(y+j)*N + x];
}

/* (ข) ใช้ไทล์ [32][32] — อ่านตามแนวตั้งของไทล์ */
__global__ void transposeTile(int N, const float *in, float *out) {
    __shared__ float tile[TILE][TILE];
    int x = blockIdx.x*TILE + threadIdx.x;
    int y = blockIdx.y*TILE + threadIdx.y;
    for (int j = 0; j < TILE; j += ROWS)
        tile[threadIdx.y + j][threadIdx.x] = in[(y+j)*N + x];
    __syncthreads();
    x = blockIdx.y*TILE + threadIdx.x;
    y = blockIdx.x*TILE + threadIdx.y;
    for (int j = 0; j < TILE; j += ROWS)
        out[(y+j)*N + x] = tile[threadIdx.x][threadIdx.y + j];   /* แนวตั้ง */
}

/* (ค) ไทล์เดียวกัน เพิ่ม padding หนึ่งคอลัมน์ */
__global__ void transposeTilePad(int N, const float *in, float *out) {
    __shared__ float tile[TILE][TILE + 1];        /* <-- ต่างกันแค่ตรงนี้ */
    int x = blockIdx.x*TILE + threadIdx.x;
    int y = blockIdx.y*TILE + threadIdx.y;
    for (int j = 0; j < TILE; j += ROWS)
        tile[threadIdx.y + j][threadIdx.x] = in[(y+j)*N + x];
    __syncthreads();
    x = blockIdx.y*TILE + threadIdx.x;
    y = blockIdx.x*TILE + threadIdx.y;
    for (int j = 0; j < TILE; j += ROWS)
        out[(y+j)*N + x] = tile[threadIdx.x][threadIdx.y + j];
}

static float timeIt(int which, int N, const float *d_in, float *d_out) {
    dim3 blk(TILE, ROWS), grd(N/TILE, N/TILE);
    cudaEvent_t s, e; float ms = 0.0f;
    CUDA_CHECK(cudaEventCreate(&s)); CUDA_CHECK(cudaEventCreate(&e));
    for (int r = 0; r < 21; r++) {
        if (r == 1) CUDA_CHECK(cudaEventRecord(s));
        if      (which == 0) transposeNaive  <<<grd, blk>>>(N, d_in, d_out);
        else if (which == 1) transposeTile   <<<grd, blk>>>(N, d_in, d_out);
        else                 transposeTilePad<<<grd, blk>>>(N, d_in, d_out);
    }
    CUDA_CHECK(cudaEventRecord(e));
    CUDA_CHECK(cudaEventSynchronize(e));
    CUDA_CHECK(cudaEventElapsedTime(&ms, s, e));
    CUDA_CHECK(cudaGetLastError());
    return ms / 20.0f;
}

int main(int argc, char **argv) {
    int N = (argc > 1) ? atoi(argv[1]) : 4096;
    size_t bytes = (size_t)N * N * sizeof(float);
    float *h_in = (float*)malloc(bytes), *h_out = (float*)malloc(bytes);
    for (int i = 0; i < N*N; i++) h_in[i] = (float)i;

    float *d_in, *d_out;
    CUDA_CHECK(cudaMalloc(&d_in, bytes)); CUDA_CHECK(cudaMalloc(&d_out, bytes));
    CUDA_CHECK(cudaMemcpy(d_in, h_in, bytes, cudaMemcpyHostToDevice));

    const char *name[3] = {"naive (ไม่ใช้ shared)", "tile [32][32]      ",
                           "tile [32][33] padded"};
    float t[3];
    for (int w = 0; w < 3; w++) {
        t[w] = timeIt(w, N, d_in, d_out);
        CUDA_CHECK(cudaMemcpy(h_out, d_out, bytes, cudaMemcpyDeviceToHost));
        int ok = 1;
        for (int i = 0; i < N && ok; i++)
            for (int j = 0; j < N; j++)
                if (h_out[j*N + i] != h_in[i*N + j]) { ok = 0; break; }
        /* ย้ายข้อมูล 2 ครั้ง (อ่าน + เขียน) => แบนด์วิดท์ที่ทำได้ */
        double gb = 2.0 * bytes / 1e9;
        printf("  %s : %7.3f ms   %6.1f GB/s   %s\n",
               name[w], t[w], gb / (t[w]/1e3), ok ? "[ผ่าน]" : "[ไม่ผ่าน]");
    }
    printf("\n  อัตราส่วน [32][32] / [32][33] = %.2f เท่า\n", t[1] / t[2]);

    free(h_in); free(h_out);
    CUDA_CHECK(cudaFree(d_in)); CUDA_CHECK(cudaFree(d_out));
    return 0;
}

Overwriting transpose.cu


In [20]:
!nvcc -O3 -arch={ARCH} transpose.cu -o transpose
!./transpose 4096

  naive (ไม่ใช้ shared) :   1.517 ms     88.5 GB/s   [ผ่าน]
  tile [32][32]       :   0.798 ms    168.1 GB/s   [ผ่าน]
  tile [32][33] padded :   0.689 ms    194.7 GB/s   [ผ่าน]

  อัตราส่วน [32][32] / [32][33] = 1.16 เท่า


**บันทึกผลและตอบคำถาม**

(ก) เติมตารางที่ 1 ด้านล่างให้ครบ

(ข) ในเวอร์ชัน `[32][32]` บรรทัด `tile[threadIdx.x][threadIdx.y + j]` — เธรด 32 ตัวใน warp เดียวกัน
อ่าน bank ใดบ้าง จงคำนวณ `bank = (address/4) % 32` ให้เห็นเป็นตัวเลข

_ตอบ:_ `bank = (tx×32 + ty+j) % 32 = (ty+j) % 32` จึงได้ bank เดิมซ้ำ 32 ตัว เช่น `0,0,...,0` และเกิด 32-way conflict

(ค) เมื่อเปลี่ยนเป็น `[32][33]` bank ที่เธรดแต่ละตัวอ่านเปลี่ยนไปอย่างไร แลกมาด้วยหน่วยความจำกี่ไบต์ต่อ block

_ตอบ:_ `bank = (tx×33 + ty+j) % 32` ทำให้กระจายครบ 32 bank จึงไม่ conflict โดยใช้หน่วยความจำเพิ่ม `32×4 = 128` ไบต์ต่อ block


---
### ส่วนที่ 4 — Launch tuning

สองเครื่องมือ: อ่านสิ่งที่คอมไพเลอร์บอก แล้วจึงวัด

**4.1 `-Xptxas -v` บอกว่า kernel ของท่านใช้ทรัพยากรเท่าใด**


In [21]:
!nvcc -O3 -arch={ARCH} -Xptxas -v -c matmul_axis.cu -o /tmp/matmul_axis.o 2>&1 | grep -A2 -E "Function|registers"

ptxas info    : Function properties for _Z14tiled32SwappediPKfS0_Pf
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 41 registers, used 1 barriers, 8192 bytes smem, 384 bytes cmem[0]
ptxas info    : Compile time = 8.401 ms
ptxas info    : Compiling entry function '_Z7tiled32iPKfS0_Pf' for 'sm_75'
ptxas info    : Function properties for _Z7tiled32iPKfS0_Pf
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 41 registers, used 1 barriers, 8192 bytes smem, 384 bytes cmem[0]
ptxas info    : Compile time = 7.379 ms


**4.2 Block size sweep**

kernel ที่ใช้คือ SAXPY จาก Lab 3 ซึ่งเป็น memory-bound ล้วน ทำให้เห็นผลของ block size ชัดเจน
โดยไม่มีเรื่อง shared memory มาปน


In [22]:
%%writefile sweep.cu
#include <cstdio>
#include <cstdlib>

#define CUDA_CHECK(call)                                        \
  do {                                                          \
    cudaError_t err = (call);                                   \
    if (err != cudaSuccess) {                                   \
      fprintf(stderr, "CUDA %s:%d: %s\n", __FILE__, __LINE__,   \
              cudaGetErrorString(err));                         \
      exit(EXIT_FAILURE);                                       \
    }                                                           \
  } while (0)

__global__ void saxpy(int n, float a, const float *x, float *y) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = a * x[i] + y[i];
}

int main() {
    const int n = 1 << 26;                 /* 67 ล้านตัว = 268 MB ต่อ array */
    const size_t bytes = (size_t)n * sizeof(float);
    float *dx, *dy;
    CUDA_CHECK(cudaMalloc(&dx, bytes)); CUDA_CHECK(cudaMalloc(&dy, bytes));
    CUDA_CHECK(cudaMemset(dx, 0, bytes)); CUDA_CHECK(cudaMemset(dy, 0, bytes));

    int sizes[] = {32, 64, 96, 128, 256, 512, 1024};
    printf("  block   grid      เวลา (ms)    GB/s    occupancy (blocks/SM)\n");
    for (int s = 0; s < 7; s++) {
        int blk = sizes[s], grd = (n + blk - 1) / blk;
        int maxBlocks = 0;
        CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(
                       &maxBlocks, saxpy, blk, 0));
        cudaEvent_t e0, e1; float ms = 0.0f;
        CUDA_CHECK(cudaEventCreate(&e0)); CUDA_CHECK(cudaEventCreate(&e1));
        for (int r = 0; r < 11; r++) {
            if (r == 1) CUDA_CHECK(cudaEventRecord(e0));
            saxpy<<<grd, blk>>>(n, 2.0f, dx, dy);
        }
        CUDA_CHECK(cudaEventRecord(e1));
        CUDA_CHECK(cudaEventSynchronize(e1));
        CUDA_CHECK(cudaEventElapsedTime(&ms, e0, e1));
        CUDA_CHECK(cudaGetLastError());
        ms /= 10.0f;
        double gb = 12.0 * n / 1e9;        /* อ่าน x, อ่าน y, เขียน y = 12 ไบต์ */
        printf("  %5d %7d   %9.3f  %7.1f    %d blocks (%d threads)\n",
               blk, grd, ms, gb / (ms/1e3), maxBlocks, maxBlocks * blk);
        CUDA_CHECK(cudaEventDestroy(e0)); CUDA_CHECK(cudaEventDestroy(e1));
    }
    CUDA_CHECK(cudaFree(dx)); CUDA_CHECK(cudaFree(dy));
    return 0;
}

Overwriting sweep.cu


In [23]:
!nvcc -O3 -arch={ARCH} sweep.cu -o sweep && ./sweep

  block   grid      เวลา (ms)    GB/s    occupancy (blocks/SM)
     32 2097152       6.661    120.9    16 blocks (512 threads)
     64 1048576       3.081    261.4    16 blocks (1024 threads)
     96  699051       3.042    264.8    10 blocks (960 threads)
    128  524288       3.036    265.3    8 blocks (1024 threads)
    256  262144       3.030    265.8    4 blocks (1024 threads)
    512  131072       3.034    265.4    2 blocks (1024 threads)
   1024   65536       3.048    264.2    1 blocks (1024 threads)


**บันทึกผลและตอบคำถาม**

(ก) block size ใดเร็วที่สุด และเร็วกว่าค่าที่ช้าที่สุดกี่เปอร์เซ็นต์

_ตอบ:_ block 256 เร็วที่สุด และเร็วกว่า block 32 ซึ่งช้าที่สุด `((6.661−3.030)/6.661)×100 = 54.51%`

(ข) `tiled32` และ `tiled32Swapped` ใช้ register และ shared memory เท่ากันหรือไม่
ถ้าเท่ากัน เหตุใดเวลาจึงต่างกันมาก ข้อสรุปนี้บอกอะไรเกี่ยวกับ occupancy

_ตอบ:_ ใช้ register และ shared memory เท่ากัน (8192 ไบต์) แต่แบบสลับแกนอ่าน global memory แบบ stride N จึงช้ากว่า แสดงว่า occupancy เท่ากันไม่ได้หมายความว่าจะเร็วเท่ากัน


### ตารางที่ 1 — สรุปผลการวัด (เติมให้ครบทุกช่อง)

| # | Kernel | ขนาด | เวลา (ms) | TFLOPS หรือ GB/s | speedup เทียบ baseline |
|---|---|---|---|---|---|
| 0 | matmul naive (baseline จาก Lab 3) | N = 2048 | 37.815 | 0.4543 TFLOPS | 1.00× |
| 1 | matmul tiled T = 16 | N = 2048 | 22.255 | 0.772 TFLOPS | 1.70× |
| 2 | matmul tiled T = 32 | N = 2048 | 16.978 | 1.012 TFLOPS | 2.23× |
| 3 | matmul tiled T = 32 สลับแกน | N = 2048 | 111.043 | 0.155 TFLOPS | 0.34× |
| 4 | transpose naive | N = 4096 | 1.517 | 88.5 GB/s | — |
| 5 | transpose tile [32][32] | N = 4096 | 0.798 | 168.1 GB/s | — |
| 6 | transpose tile [32][33] | N = 4096 | 0.689 | 194.7 GB/s | — |

> แถว 1–3 รายงานเป็น TFLOPS · แถว 4–6 รายงานเป็น GB/s


---
## 4. คำถามวิเคราะห์

**Q1.** ที่ N = 2048 เวอร์ชัน tiled T = 32 ของท่านทำได้กี่ % ของ peak FP32 จาก Lab 2
และ arithmetic intensity ของ tiled T = 32 เป็นเท่าใด (แสดงวิธีคิด) จากค่านี้ kernel ยังเป็น
memory-bound หรือกลายเป็น compute-bound แล้ว

_ตอบ:_ `AI = 2N³ / [(2N³/32)×4] = 8 FLOP/byte` ต่ำกว่า ridge 33.8 จึงยังเป็น memory-bound และ `%peak = (1.012/8.14)×100 = 12.43%`

**Q2.** เหตุใด T = 32 จึงเร็วกว่า T = 16 ทั้งที่ทั้งคู่ก็ใช้ shared memory เหมือนกัน
ให้ตอบด้วยจำนวนครั้งที่ข้อมูลหนึ่งช่องถูกใช้ซ้ำ และปริมาณ global load ทั้งหมด (สูตร `2N³/T`)

_ตอบ:_ T=16 ใช้ข้อมูลซ้ำ 16 ครั้งและโหลด `N³/8` float ส่วน T=32 ใช้ซ้ำ 32 ครั้งและโหลด `N³/16` float จึงลด global load ลงครึ่งหนึ่ง

**Q3.** ถ้าลบ `__syncthreads()` ตัวที่สอง (ตัวที่อยู่ท้ายลูป) ออก โปรแกรมจะผิดอย่างไร
เหตุใดจึงมักยังให้ผลถูกต้องเมื่อ N เล็ก แต่ผิดเมื่อ N ใหญ่

_ตอบ:_ warp ที่เร็วอาจเขียนไทล์ใหม่ทับขณะที่ warp อื่นยังอ่านไทล์เดิม ทำให้เกิด race condition เมื่อ N เล็กอาจบังเอิญถูก แต่ N ใหญ่มีรอบมากจึงเกิดความผิดพลาดง่ายขึ้น

**Q4.** จากผลทั้งหมดในแลปนี้ ถ้าท่านมีเวลาอีกหนึ่งชั่วโมงเพื่อปรับ matmul ให้เร็วขึ้นอีก
ท่านจะทำอะไรเป็นอันดับแรก และจะ **วัด** อย่างไรว่าสิ่งนั้นได้ผล

_ตอบ:_ จะทำ register tiling ให้หนึ่งเธรดคำนวณหลาย output แล้ววัด N=2048 ด้วย CUDA events หลัง warm-up ตรวจความถูกต้อง และเทียบเวลา TFLOPS กับ tiled32 เดิม


---
## 5. การเตรียมไฟล์ส่งงาน

รันเซลล์ถัดไปเพื่อบีบไฟล์ `.cu` ทั้งหมดแล้วดาวน์โหลด จากนั้นอัปโหลดพร้อมโน้ตบุ๊กนี้ผ่าน LMS

> **ก่อนรัน** แก้ `sid` ในเซลล์ถัดไปให้เป็นรหัสนักศึกษาของท่าน
> ชื่อไฟล์ที่ผิดจะทำให้ระบบตรวจงานหาไฟล์ไม่เจอ


In [24]:
import os
sid = "6730614023"        # TODO: แก้เป็นรหัสนักศึกษาของท่าน
!zip -q Lab4_{sid}.zip *.cu
print("ไฟล์ที่บีบแล้ว:", os.path.getsize(f"Lab4_{sid}.zip"), "bytes")
try:
    from google.colab import files
    files.download(f"Lab4_{sid}.zip")
except Exception as e:
    print("ดาวน์โหลดเองจากแถบไฟล์ด้านซ้าย:", e)

ไฟล์ที่บีบแล้ว: 6459 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 6. รายการสิ่งที่ต้องส่ง

- [ ] โน้ตบุ๊ก `Lab4_<รหัสนักศึกษา>.ipynb` รันครบทุกเซลล์พร้อม output
- [ ] ไฟล์ `matmul_tiled.cu`, `matmul_axis.cu`, `transpose.cu`, `sweep.cu` (บีบรวมเป็น .zip)
- [ ] ทุก kernel ผ่านการตรวจความถูกต้อง `[ผ่าน]` ก่อนนำเวลาไปรายงาน
- [ ] ตารางที่ 1 ครบทุกช่อง พร้อมหน่วยที่ถูกต้อง (TFLOPS สำหรับ matmul · GB/s สำหรับ transpose)
- [ ] คำตอบของส่วนที่ 2 (ก)(ข) · ส่วนที่ 3 (ข)(ค) · ส่วนที่ 4 (ก)(ข)
- [ ] คำตอบคำถามวิเคราะห์ Q1–Q4

## 7. เกณฑ์การให้คะแนน

| องค์ประกอบ | คะแนน | เกณฑ์การพิจารณา |
|---|---|---|
| ความถูกต้องของ kernel | 30 | tiled matmul ผ่านทั้งสาม N · วาง `__syncthreads()` ถูกทั้งสองจุด · ตรวจ error ครบ |
| การวัดผล | 30 | ใช้ CUDA events + warm-up · หน่วยถูกต้อง · เทียบกับ baseline ของตนเอง |
| ผลการปรับแต่ง | 20 | ส่วนที่ 2 และ 3 ให้ทิศทางผลถูกต้อง พร้อมคำอธิบายเชิงฮาร์ดแวร์ |
| คำตอบคำถามวิเคราะห์ | 20 | Q1–Q4 ข้อละ 5 โดยเฉพาะ Q1 และ Q3 ต้องแสดงวิธีคิด |
| **รวม** | **100** | |
